# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [42]:
import os
import numpy as np
import pandas as pd


In [43]:
!git clone https://github.com/jenilrupareliya5150-bit/FlyRankAi-ml-Track.git

Cloning into 'FlyRankAi-ml-Track'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 198 (delta 92), reused 89 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 2.68 MiB | 4.21 MiB/s, done.
Resolving deltas: 100% (92/92), done.


In [44]:
%cd FlyRankAi-ml-Track

/content/FlyRankAi-ml-Track/FlyRankAi-ml-Track/FlyRankAi-ml-Track


In [45]:
df=pd.read_csv(r"data/raw/content_refresh_anonymized.csv")

In [46]:
df.sample(5)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
8280,content_94600dab7594,client_bbb965ab0c,0.0,0.00,LOW,0.00,keyword article,informational,2689.0,20018.0,...,15000-25000,0.00,38.6,0.00,0.00,0.0,moderate,page_3_5,down,-39.8
10490,content_fba23b9a8913,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,1448.0,9172.0,...,8000-15000,0.83,13.6,0.85,1.83,0.0,good,striking,stable,1.0
7317,content_662d57b2d94d,client_3fdba35f04,30.0,0.01,LOW,0.00,keyword article,informational,1281.0,7831.0,...,<8000,0.21,10.5,0.00,25.00,0.0,moderate,striking,up,23.3
19569,content_38252e54b185,client_349c41201b,20.0,0.85,HIGH,0.64,keyword article,commercial,3852.0,24992.0,...,15000-25000,0.27,45.0,0.00,0.00,0.0,good,page_3_5,down,-54.7
21,content_9d548144b06d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3455.0,21571.0,...,15000-25000,0.00,12.6,0.00,0.00,0.0,low,striking,stable,11.1


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The model ranks content items by their predicted probability of declining.
The highest-ranked items are candidates for human review and possible refresh.

Actions are recommendations, not automatic decisions. Each recommendation includes
a reason code based on the signals used by the model.

In [47]:
# Create the target used by the validated Week-5 model

df["declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
print(df["declining_label"].value_counts())

Target distribution:
declining_label
1    16262
0    13738
Name: count, dtype: int64


In [48]:
# Columns that should not be used as model features

exclude_cols = [
    "declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

feature_cols = [
    col for col in df.columns
    if col not in exclude_cols
]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 40
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [49]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        df["declining_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

client_overlap = (
    set(train_df["client_id"]) &
    set(test_df["client_id"])
)

print("Client overlap:", len(client_overlap))

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

X_train = train_df[feature_cols]
y_train = train_df["declining_label"]

X_test = test_df[feature_cols]
y_test = test_df["declining_label"]

numeric_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [51]:
# Predict probability of declining

test_df["declining_probability"] = model.predict_proba(
    X_test
)[:, 1]

# Rank highest-risk content first


ranked_queue = test_df.sort_values(
    "declining_probability",
    ascending=False
).copy()


ranked_queue["rank"] = range(1, len(ranked_queue) + 1)

print("Top 20 ranked content:")
display(
    ranked_queue[
        [
            "rank",
            "content_id",
            "client_id",
            "declining_probability"
        ]
    ].sample(20)
)

Top 20 ranked content:


,rank,content_id,client_id,declining_probability
22595,1730,content_23c380f126ad,client_8527a891e2,6.816173e-01
29612,445,content_07f7cbf34eef,client_f369cb89fc,9.186521e-01
2827,5244,content_5ef5a39eade1,client_e629fa6598,2.129092e-01
19909,3006,content_c83476e39f7d,client_4e07408562,5.595965e-01
27430,2423,content_b39885a157fd,client_4e07408562,6.159397e-01
13215,433,content_98aa0aecb1d9,client_8527a891e2,9.248590e-01
19162,4054,content_292efd8c2a22,client_e629fa6598,4.231997e-01
23219,3216,content_35548a435e63,client_4e07408562,5.378244e-01
21018,1675,content_ea2dcd45a28c,client_f369cb89fc,6.856634e-01
9141,3049,content_9fc41e4ea334,client_8527a891e2,5.565575e-01


In [52]:
def assign_action(probability):
    if probability >= 0.80:
        return "Human review - consider refresh"
    elif probability >= 0.60:
        return "Monitor"
    else:
        return "No immediate action"


ranked_queue["recommended_action"] = (
    ranked_queue["declining_probability"]
    .apply(assign_action)
)

display(
    ranked_queue[
        [
            "rank",
            "content_id",
            "declining_probability",
            "recommended_action"
        ]
    ].sample(20)
)

,rank,content_id,declining_probability,recommended_action
27783,4666,content_6cac3cdb43ec,3.202239e-01,No immediate action
1114,2148,content_e053030b589a,6.412515e-01,Monitor
11058,5086,content_1a0ba11dff95,2.496059e-01,No immediate action
2704,1894,content_51295f0b7748,6.655140e-01,Monitor
29656,2582,content_5ace8eec26ba,6.007293e-01,Monitor
22372,4060,content_8883df6fc98b,4.225908e-01,No immediate action
10688,6132,content_e04b14da018b,9.270940e-23,No immediate action
23769,5979,content_ab943e8575eb,1.831661e-06,No immediate action
22146,4311,content_aa27d7969080,3.804953e-01,No immediate action
17258,3078,content_e59846c037ae,5.534581e-01,No immediate action


In [54]:
def reason_code(row):
    reasons = []

    if row["trend_direction"] == "down":
        reasons.append("trend_down")

    if pd.notna(row["engagement_rate"]) and row["engagement_rate"] < 0.05:
        reasons.append("low_engagement")

    if pd.notna(row["scroll_rate"]) and row["scroll_rate"] < 0.50:
        reasons.append("low_scroll")

    if pd.notna(row["impressions_90d"]) and row["impressions_90d"] == 0:
        reasons.append("no_recent_impressions")

    if len(reasons) == 0:
        reasons.append("model_rank")

    return ", ".join(reasons[:3])


ranked_queue["reason_code"] = ranked_queue.apply(
    reason_code,
    axis=1
)

In [55]:
display(
    ranked_queue[
        [
            "rank",
            "content_id",
            "declining_probability",
            "recommended_action",
            "reason_code"
        ]
    ].head(20)
)

,rank,content_id,declining_probability,recommended_action,reason_code
29660,1,content_6926821766df,1.0,Human review - consider refresh,trend_down
14949,2,content_bf658584d14d,1.0,Human review - consider refresh,trend_down
13842,3,content_c97ce8d47eef,1.0,Human review - consider refresh,trend_down
13705,4,content_25f7bcf2a206,1.0,Human review - consider refresh,"trend_down, low_engagement"
27178,5,content_453722754fea,1.0,Human review - consider refresh,"trend_down, low_engagement"
5338,6,content_d7175187ff12,1.0,Human review - consider refresh,trend_down
14226,7,content_65d9331f55fb,1.0,Human review - consider refresh,trend_down
14638,8,content_bfeab0d95550,1.0,Human review - consider refresh,"trend_down, low_engagement"
482,9,content_39881853ef0c,1.0,Human review - consider refresh,trend_down
16788,10,content_8094eccaeb0a,1.0,Human review - consider refresh,"trend_down, low_engagement"


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This action playbook is intended for content teams to prioritize webpages for human review using the model's estimated probability of declining performance.

The ranked queue helps identify which content may deserve attention first. A higher declining probability is a prioritization signal, not proof that a webpage should be refreshed.

The output should not be used to automatically refresh, rewrite, delete, or publish content. A human should review the webpage and its context before taking action.

The model is limited by the historical data and features used to train and validate it. Its predictions should therefore be treated as decision-support rather than a guarantee of future performance.

In [61]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before taking any action, a human reviewer should check the webpage's recent performance, content context, and whether the model's declining signal is supported by the available evidence.

The reviewer should consider whether the page has enough recent data, whether the decline is meaningful, and whether there are other reasons for the observed change.

The model should never automatically rewrite, delete, publish, or refresh a webpage. It should also not make a final decision about content quality or guarantee that a refresh will improve performance.

The model output is only a prioritization and decision-support signal. Final actions remain the responsibility of a human reviewer.

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The action playbook should be reviewed if the model's performance changes or if the underlying data distribution changes significantly.

Possible retrain or review triggers include:
- A noticeable drop in Precision@50 or ROC-AUC compared with previous evaluations.
- Changes in content patterns, search behavior, or feature distributions.
- New data becoming available that better represents current content performance.
- Consistent feedback from human reviewers that the recommendations are inaccurate or not useful.

The model should be re-evaluated on an honest client-grouped validation split before replacing the current version.

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked action queue is exported so that the paper and later analysis can reuse the same model output. The export contains the content identifier, declining probability, recommended action, and ranking information.

The exported queue is a record of the model's decision-support output and does not represent an automatic content decision. Any figures or summary metrics used later should also be generated from the validated results in this notebook.

In [62]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output folder:", output_dir)

Output folder: work/outputs


In [63]:
output_file = output_dir / "ranked_action_queue.csv"

ranked_queue.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: work/outputs/ranked_action_queue.csv


In [64]:
print(ranked_queue.shape)
display(ranked_queue.head())

(6163, 49)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,declining_label,declining_probability,rank,recommended_action,reason_code
29660,content_6926821766df,client_f369cb89fc,10.0,0.12,LOW,0.00,keyword article,informational,2615.0,18368.0,...,0.0,excellent,page_1,down,-66.6,1,1.0,1,Human review - consider refresh,trend_down
14949,content_bf658584d14d,client_f369cb89fc,40.0,0.36,MEDIUM,0.03,keyword article,informational,2657.0,18704.0,...,0.0,excellent,page_1,down,-76.3,1,1.0,2,Human review - consider refresh,trend_down
13842,content_c97ce8d47eef,client_f369cb89fc,10.0,0.00,LOW,0.00,keyword article,informational,2400.0,17312.0,...,0.0,good,page_1,down,-62.7,1,1.0,3,Human review - consider refresh,trend_down
13705,content_25f7bcf2a206,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2929.0,18968.0,...,0.0,good,page_3_5,down,-66.7,1,1.0,4,Human review - consider refresh,"trend_down, low_engagement"
27178,content_453722754fea,client_f369cb89fc,10.0,0.00,LOW,0.00,keyword article,informational,2700.0,18723.0,...,0.0,excellent,page_1,down,-52.9,1,1.0,5,Human review - consider refresh,"trend_down, low_engagement"


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.